# Маскирование персональных данных и синтетическая замена

**Задача:** дообучить seq2seq-модель находить персональные данные в тексте, заменять их типизированными тегами и при необходимости подставлять синтетические значения.

**Стек:** Transformers, Hugging Face Datasets, PyTorch, Faker.

Пайплайн состоит из трёх частей: fine-tuning T5, оценка качества тегов и безопасная демонстрация синтетической замены.


In [ ]:
from collections import Counter
import re

import numpy as np
import torch
from datasets import load_dataset
from faker import Faker
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

SEED = 42
MODEL_NAME = "t5-small"
TRAIN_SIZE = 10_000
TEST_SIZE = 1_000
MAX_LENGTH = 256
PREFIX = "mask pii: "
OUTPUT_DIR = "./artifacts/pii_masking_t5"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

set_seed(SEED)
fake = Faker("en_US")
Faker.seed(SEED)


In [ ]:
dataset = load_dataset("ai4privacy/pii-masking-200k", split="train")
required_columns = {"source_text", "target_text"}
if missing := required_columns - set(dataset.column_names):
    raise ValueError(f"В датасете отсутствуют колонки: {sorted(missing)}")

subset_size = min(TRAIN_SIZE + TEST_SIZE, len(dataset))
subset = dataset.shuffle(seed=SEED).select(range(subset_size))
split = subset.train_test_split(test_size=min(TEST_SIZE, subset_size // 5), seed=SEED)
train_dataset = split["train"]
test_dataset = split["test"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)

print(f"Train: {len(train_dataset):,}; test: {len(test_dataset):,}")


In [ ]:
def preprocess(batch):
    inputs = [PREFIX + str(text) for text in batch["source_text"]]
    targets = [str(text) for text in batch["target_text"]]
    return tokenizer(
        inputs,
        text_target=targets,
        max_length=MAX_LENGTH,
        truncation=True,
    )


tokenized_train = train_dataset.map(
    preprocess,
    batched=True,
    remove_columns=train_dataset.column_names,
)
tokenized_test = test_dataset.map(
    preprocess,
    batched=True,
    remove_columns=test_dataset.column_names,
)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=5e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)


In [ ]:
def mask_pii(text: str) -> str:
    encoded = tokenizer(
        PREFIX + str(text),
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(DEVICE)
    with torch.inference_mode():
        generated = model.generate(
            **encoded,
            num_beams=4,
            max_new_tokens=MAX_LENGTH,
            early_stopping=True,
        )
    return tokenizer.decode(generated[0], skip_special_tokens=True)


TAG_PATTERN = re.compile(r"\[[A-Z][A-Z0-9_]*\]")


def tag_counts(text: str) -> Counter:
    return Counter(TAG_PATTERN.findall(str(text)))


def multiset_overlap(left: Counter, right: Counter) -> int:
    return sum((left & right).values())


In [ ]:
evaluation_size = min(200, len(test_dataset))
true_positive = false_positive = false_negative = 0
exact_matches = 0

for row in test_dataset.select(range(evaluation_size)):
    prediction = mask_pii(row["source_text"])
    true_tags = tag_counts(row["target_text"])
    predicted_tags = tag_counts(prediction)
    overlap = multiset_overlap(true_tags, predicted_tags)

    true_positive += overlap
    false_positive += sum(predicted_tags.values()) - overlap
    false_negative += sum(true_tags.values()) - overlap
    exact_matches += prediction.strip() == str(row["target_text"]).strip()

precision = true_positive / max(true_positive + false_positive, 1)
recall = true_positive / max(true_positive + false_negative, 1)
f1 = 2 * precision * recall / max(precision + recall, 1e-12)

print(f"Tag precision: {precision:.4f}")
print(f"Tag recall:    {recall:.4f}")
print(f"Tag F1:        {f1:.4f}")
print(f"Exact match:   {exact_matches / evaluation_size:.4f}")


In [ ]:
def synthetic_value(tag: str) -> str:
    normalized = tag.strip("[]").upper()
    if "NAME" in normalized:
        return fake.name()
    if "EMAIL" in normalized:
        return fake.email()
    if "PHONE" in normalized or "IMEI" in normalized:
        return fake.phone_number()
    if any(part in normalized for part in ["CITY", "LOCATION", "STATE", "ADDRESS"]):
        return fake.address().replace("\n", ", ")
    if "COMPANY" in normalized or "ORG" in normalized:
        return fake.company()
    if "JOB" in normalized:
        return fake.job()
    if "DATE" in normalized:
        return fake.date()
    return f"<{normalized.lower()}>"


def replace_tags_with_synthetic_data(masked_text: str) -> str:
    return TAG_PATTERN.sub(lambda match: synthetic_value(match.group(0)), masked_text)


In [ ]:
for index in [0, 1, 2]:
    source = test_dataset[index]["source_text"]
    masked = mask_pii(source)
    synthetic = replace_tags_with_synthetic_data(masked)
    print(f"Пример {index + 1}")
    print("Исходный текст:", source)
    print("Маскирование:  ", masked)
    print("Синтетика:     ", synthetic)
    print()


## Ограничения и безопасное применение

Сопоставление только типов тегов не проверяет точные границы сущностей, поэтому в продакшен-оценке нужны span-level precision/recall. Синтетическая подстановка предназначена для тестовых данных и демонстраций; она не должна использоваться для восстановления реальных персональных данных.
